In [ ]:
#| hide
!pip install -Uqq nixtla

# Plan Retail Promotions with Coupled Simulation

A store plans a 15% price cut on one product for the next four weeks, and the category manager needs order quantities for three related products: the promoted product itself, an alternative product customers might have bought instead, and a related product often bought alongside it.

This notebook uses coupled simulation to answer two questions:

1. How could a promotion change demand across a product group?
2. How often could several products need replenishment at the same time?

With coupled simulation, one `sample_id` represents one possible future for the whole product group.

## Step 1: Import packages

Import the required packages and initialize a Nixtla client.

In [ ]:
import numpy as np
import pandas as pd

from nixtla import NixtlaClient

In [ ]:
nixtla_client = NixtlaClient(
    # Defaults to os.environ["NIXTLA_API_KEY"]
    api_key="my_api_key_provided_by_nixtla"
)

## Step 2: Load the product-group data

The example uses three years of daily sales for the three products. Every row carries all three prices, so when the promoted product's price changes, the demand simulation for every product can respond.

The data is generated rather than observed, so the product relationships are known exactly: the alternative product is a substitute (it loses demand when the promoted product gets cheaper) and the related product is a complement (it gains demand alongside the promoted product). An unobserved store-footfall factor lifts all three products on the same day; it is never given to the model, so it survives as demand that moves together across the products — which is exactly what coupling has to reproduce.

In [ ]:
rng = np.random.default_rng(11)
n = 1095          # three years of daily history
h = 28            # the promotion window
t = np.arange(n + h)

# Each product runs its own price schedule; the three do not move together.
promoted_price = (
    6.00
    + 0.60 * np.sin(2 * np.pi * t / 97)
    + 0.30 * np.sin(2 * np.pi * t / 29)
    + rng.normal(0, 0.30, n + h)
)
alternative_price = (
    9.00
    + 0.85 * np.sin(2 * np.pi * t / 73 + 1.1)
    + 0.40 * np.sin(2 * np.pi * t / 23)
    + rng.normal(0, 0.36, n + h)
)
related_price = (
    4.50
    + 0.45 * np.sin(2 * np.pi * t / 113 + 2.3)
    + 0.22 * np.sin(2 * np.pi * t / 31)
    + rng.normal(0, 0.24, n + h)
)

# Store footfall drifts through busy and quiet stretches, lifting every product
# on the same day. It is never given to the model.
shock = rng.normal(0, 1, n + h)
footfall = np.zeros(n + h)
for i in range(1, n + h):
    footfall[i] = 0.92 * footfall[i - 1] + shock[i]
footfall = 1 + 0.059 * footfall
weekly = np.sin(2 * np.pi * t / 7)

# The product relationships are planted here.
promoted_demand = (
    40 * footfall
    - 5.3 * (promoted_price - 6.0)     # own-price effect
    + 2.0 * (alternative_price - 9.0)
    + 3.0 * weekly
    + rng.normal(0, 1.6, n + h)
)
alternative_demand = (
    55 * footfall
    + 7.0 * (promoted_price - 6.0)     # substitute: cheaper promoted, fewer sales
    - 4.0 * (alternative_price - 9.0)
    + 3.0 * weekly
    + rng.normal(0, 2.0, n + h)
)
related_demand = (
    30 * footfall
    - 5.0 * (promoted_price - 6.0)     # complement: cheaper promoted, more sales
    - 3.5 * (related_price - 4.5)
    + 2.0 * weekly
    + rng.normal(0, 1.4, n + h)
)

dates = pd.date_range("2023-01-01", periods=n + h, freq="D")
prices = {
    "promoted_price": promoted_price,
    "alternative_price": alternative_price,
    "related_price": related_price,
}
demands = {
    "Promoted product": promoted_demand,
    "Alternative product": alternative_demand,
    "Related product": related_demand,
}

history, future = [], []
for label, demand in demands.items():
    history.append(
        pd.DataFrame(
            {
                "unique_id": label,
                "ds": dates[:n],
                "y": np.maximum(demand[:n], 0).round(),
                **{name: values[:n] for name, values in prices.items()},
            }
        )
    )
    future.append(
        pd.DataFrame(
            {
                "unique_id": label,
                "ds": dates[n:],
                **{name: values[n:] for name, values in prices.items()},
            }
        )
    )

df = pd.concat(history, ignore_index=True)
current_X_df = pd.concat(future, ignore_index=True)

df.head()

## Step 3: Define the promotion

The next 28 days carry regular prices. Create a second plan with the promoted product's price reduced by 15%, leaving the other prices unchanged.

In [ ]:
promotion_X_df = current_X_df.copy()
promotion_X_df["promoted_price"] *= 0.85

## Step 4: Simulate the product group under both plans

Generate 500 possible futures for all three products, first at current prices, then with the promotion.

`multivariate=True` changes two things at once: TimeGPT 2.1 forecasts the products jointly, so each product's forecast distribution can reflect the others, and the sample paths are coupled, so one `sample_id` is one future for the whole group.

In [ ]:
current_paths = nixtla_client.simulate(
    df=df,
    X_df=current_X_df,
    h=h,
    freq="D",
    n_paths=500,
    seed=42,
    model="timegpt-2.1",
    multivariate=True,
)

promotion_paths = nixtla_client.simulate(
    df=df,
    X_df=promotion_X_df,
    h=h,
    freq="D",
    n_paths=500,
    seed=42,
    model="timegpt-2.1",
    multivariate=True,
)

promotion_paths["coupled"].unique()

## Step 5: Measure lift, cannibalization, and cross-selling

Product demand cannot be negative, so clip values at zero before calculating unit totals.

In [ ]:
def path_totals(paths):
    nonnegative = paths.assign(TimeGPT=paths["TimeGPT"].clip(lower=0))
    return nonnegative.pivot_table(
        index="sample_id",
        columns="unique_id",
        values="TimeGPT",
        aggfunc="sum",
    )


current_totals = path_totals(current_paths)
promotion_totals = path_totals(promotion_paths)

impact = pd.DataFrame(
    {
        "Current prices": current_totals.median(),
        "Promotion": promotion_totals.median(),
    }
)
impact["Change"] = impact["Promotion"] - impact["Current prices"]
impact["Change (%)"] = impact["Promotion"] / impact["Current prices"] - 1

impact

The promoted product gains units, the related product gains units (cross-selling), and the alternative product loses units (cannibalization).

Looking only at the promoted product suggests a successful promotion. Looking at the complete product group shows that the alternative product loses more units than the other two gain combined.

## Step 6: Estimate shared-demand risk

To see what simulating the products *together* changes, generate the promotion paths once more with the products simulated separately.

The two runs use different seeds on purpose: with one seed, the coupled and the per-series shuffle pick their template windows from the same random state, so the first product by name can come back with identical paths in both runs.

In [ ]:
separate_paths = nixtla_client.simulate(
    df=df,
    X_df=promotion_X_df,
    h=h,
    freq="D",
    n_paths=500,
    seed=7,
    model="timegpt-2.1",
    multivariate=False,
)

separate_paths["coupled"].unique()

Now ask an operational question:

> What is the chance that at least two products experience high demand on the same day during the promotion?

For this example, "high demand" means demand above that product's 90th percentile in the separately simulated paths. The same thresholds are applied to both sets of paths.

In [ ]:
def daily_paths(paths):
    nonnegative = paths.assign(TimeGPT=paths["TimeGPT"].clip(lower=0))
    return nonnegative.pivot(
        index=["sample_id", "ds"],
        columns="unique_id",
        values="TimeGPT",
    )


separate_daily = daily_paths(separate_paths)
coupled_daily = daily_paths(promotion_paths)
high_demand = separate_daily.quantile(0.90)


def simultaneous_high_demand(daily):
    two_or_more = daily.gt(high_demand).sum(axis=1).ge(2)
    return two_or_more.groupby("sample_id").any().mean()


separate_risk = simultaneous_high_demand(separate_daily)
coupled_risk = simultaneous_high_demand(coupled_daily)

separate_risk, coupled_risk

The products share a demand driver the model never sees. Simulating each one on its own throws that shared movement away and treats busy days as independent coincidences. Coupling puts it back, and more of the futures contain a day when several products are under pressure at once.

Planning each product on its own understates how often they will need attention at the same time. That gap can affect replenishment staffing, shelf capacity, and safety-stock decisions.

## Turn the result into a retail decision

This example suggests three actions:

1. Increase inventory for the promoted product.
2. Prepare for additional related-product demand.
3. Reduce the alternative product's order or reconsider the discount if total product-group volume is the goal.

The final decision should include revenue, product margin, inventory cost, and stockout cost. Those values can be calculated for every `sample_id`, producing a distribution of profit instead of only a distribution of units.

**Note:** with real sales data, a demand response like this is an association rather than proof of a mechanism. Use experiments or basket data when you need to establish why customers changed their purchases.